In [ ]:
!pip install optuna
!pip install lightgbm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import gc
import os

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, f1_score,
                             precision_score, recall_score,
                             roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score)

warnings.filterwarnings('ignore')
print(f"LightGBM {lgb.__version__}")


In [ ]:
# ═══════════════════════════════════════════════════
# KONFIGURÁCIA
# ═══════════════════════════════════════════════════
DATA_PATH = '../Data/Export_data/forecast_10min.parquet'
SEGMENTS_PATH = '../Data/Export_data/segments.csv'
OUTPUT_DIR = '../Modely/LightGBM/lightgbm_10min'
# Cache na disku - vygenerované features sa uložia raz a potom sa len načítavajú,
# šetrí to pamäť aj čas pri ladení
CACHE_DIR = '../Modely/LightGBM/lightgbm_10min/cache'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

VOLTAGE_COLS = ['u1_norm', 'u2_norm', 'u3_norm']
CURRENT_COLS = ['i1_norm', 'i2_norm', 'i3_norm']
TARGET_COLS = VOLTAGE_COLS + CURRENT_COLS

SELECTED_HORIZONS = [6, 18, 36, 72]

# Base parametre (Optuna doplní zvyšok)
LGBM_BASE = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'verbose': -1,
    'n_jobs': -1,
    'seed': 42
}
# Maximum stromov - early stopping ich zvyčajne zastaví skôr
NUM_BOOST_ROUNDS = 500
EARLY_STOPPING = 50

# Optuna konfigurácia - tunujeme len na jednom targete a horizonte (rýchlejšie),
# najlepšie parametre potom použijeme pre všetkých 6 cieľových premenných
OPTUNA_TRIALS = 30
OPTUNA_TARGET = 'u1_norm'
OPTUNA_HORIZON = 6

ANOMALY_SIGMA = 3.0

print(f"Horizonty predikcie: {SELECTED_HORIZONS}")
print(f"Sigma pre anomáliu:  {ANOMALY_SIGMA}")


---
# ČASŤ 1: NAČÍTANIE DÁT A ROZDELENIE
---

In [ ]:
%%time
data = pd.read_parquet(DATA_PATH)
data['t_utc'] = pd.to_datetime(data['t_utc'])
# float32 namiesto float64 šetrí pamäť na polovicu
for col in data.select_dtypes(include=['float64']).columns:
    data[col] = data[col].astype('float32')

print(f"Celkovo: {len(data):,} riadkov")
print(f"Chyba=0: {(data['Chyba']==0).sum():,}, Chyba=1: {(data['Chyba']==1).sum():,}")

# ─────────────────────────────────────────────────
# KROK 1: Identifikácia segmentov
# ─────────────────────────────────────────────────
# Pre každý segment zistíme jeho stav (chyba_max=0 znamená, že celý segment je čistý)
seg_info = data.groupby(['eic', 'segment_id'], observed=True).agg(
    start_time=('t_utc', 'min'),
    end_time=('t_utc', 'max'),
    num_records=('t_utc', 'count'),
    chyba_max=('Chyba', 'max')
).reset_index()

# Rozdelenie na čisté a poruchové segmenty - sortujeme podľa času pre chronologický split
clean_segs = seg_info[seg_info['chyba_max'] == 0].sort_values('start_time').reset_index(drop=True)
error_segs = seg_info[seg_info['chyba_max'] == 1].sort_values('start_time').reset_index(drop=True)
print(f"\nKROK 1: {len(clean_segs):,} čistých segmentov, {len(error_segs):,} poruchových segmentov")

# ─────────────────────────────────────────────────
# KROK 2: 70/15/15 split ČISTÝCH segmentov (chronologicky)
# ─────────────────────────────────────────────────
# Keďže sú zoradené podľa času, train sú najstaršie a test najnovšie - tým simulujeme
# reálnu situáciu, kde model trénovaný na minulosti predikuje budúcnosť
n_clean = len(clean_segs)
cut_70 = int(n_clean * 0.70)
cut_85 = int(n_clean * 0.85)

# Sety dvojíc (eic, segment_id) - rýchly lookup pri filtrovaní dát nižšie
train_seg_keys = set(zip(clean_segs.iloc[:cut_70]['eic'], clean_segs.iloc[:cut_70]['segment_id']))
val_seg_keys = set(zip(clean_segs.iloc[cut_70:cut_85]['eic'], clean_segs.iloc[cut_70:cut_85]['segment_id']))
test_seg_keys = set(zip(clean_segs.iloc[cut_85:]['eic'], clean_segs.iloc[cut_85:]['segment_id']))
error_seg_keys = set(zip(error_segs['eic'], error_segs['segment_id']))

print(f"KROK 2 (Chyba=0): train={len(train_seg_keys)}, val={len(val_seg_keys)}, test={len(test_seg_keys)}")
print(f"         Chyba=1: error={len(error_seg_keys)}")

# ─────────────────────────────────────────────────
# KROK 3: Priradenie dát
# ─────────────────────────────────────────────────
# Filtrovanie cez sety - pre každý riadok overíme, či jeho (eic, segment_id) je v sete
seg_key = list(zip(data['eic'], data['segment_id']))
train_mask = np.array([k in train_seg_keys for k in seg_key])
val_mask = np.array([k in val_seg_keys for k in seg_key])
test_mask = np.array([k in test_seg_keys for k in seg_key])
error_mask = np.array([k in error_seg_keys for k in seg_key])

train_clean = data[train_mask].copy()
val_clean = data[val_mask].copy()
test_clean = data[test_mask].copy()
error_data = data[error_mask].copy()

# Anomaly detection = test_clean + error_data
# (model nevidel test_clean pri tréningu → prah z nich, detekcia na error_data)
anomaly_data = pd.concat([test_clean, error_data], ignore_index=True)

del seg_key, train_mask, val_mask, test_mask, error_mask, data
gc.collect()

# Súhrnná tabuľka rozdelenia
sep = '=' * 70
print(f"\n{sep}")
print(f"{'MNOŽINA':<22s} {'RIADKOV':>12s} {'Chyba=0':>12s} {'Chyba=1':>12s} {'Segmentov':>10s}")
print(sep)
for nm, df in [('train_clean', train_clean),
               ('val_clean', val_clean),
               ('test_clean', test_clean),
               ('error_data', error_data),
               ('anomaly_data', anomaly_data)]:
    c0 = (df['Chyba']==0).sum()
    c1 = (df['Chyba']==1).sum()
    n_seg = df.groupby(['eic','segment_id'], observed=True).ngroups
    print(f"  {nm:<20s} {len(df):>10,} {c0:>10,} {c1:>10,} {n_seg:>9,}")
print(sep)
print("\nForecasting train/val/test: len Chyba=0")
print("Anomaly detection: test_clean (prah) + error_data (detekcia)")


---
# ČASŤ 2: FEATURE ENGINEERING
---

In [ ]:
def create_features_and_save(df, split_name, target_cols, horizons, cache_dir):
    """Spracuj 1 split: features → numpy na disk → uvoľni RAM."""
    # Funkcia spracuje jeden split (train/val/test/anomaly), vytvorí všetky features
    # a uloží ich na disk ako numpy arrays. Potom sa pamäť uvoľní.
    # Pri viacerých splitoch by inak RAM nestíhala (tréning má rádovo milión riadkov).
    print(f"\n{'='*50}")
    print(f"{split_name}: {len(df):,} riadkov")
    
    df = df.sort_values(['eic', 'segment_id', 't_utc']).reset_index(drop=True)
    grp = df.groupby(['eic', 'segment_id'])
    
    # ─── Časové features ───
    # Hodina dňa, deň v týždni, je víkend - pomáha modelu zachytiť denné/týždenné cykly
    df['hour'] = df['t_utc'].dt.hour.astype('int8')
    df['dow'] = df['t_utc'].dt.dayofweek.astype('int8')
    df['is_weekend'] = (df['dow'] >= 5).astype('int8')
    # Cyklické kódovanie cez sin/cos - inak by sa hodina 23 a hodina 0 javili
    # ako vzdialené (23 vs 0), pričom v skutočnosti sú vedľa seba
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24).astype('float32')
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24).astype('float32')
    df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7).astype('float32')
    df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7).astype('float32')
    
    # ─── Lag features ───
    # Hodnota z minulosti (pred 1, 2, 3, 6, 12, 24, 48 krokmi).
    # shift() v rámci skupiny (eic+segment) - aby sa hodnoty nemiešali medzi rôznymi segmentmi
    print("  Lags...", end=" ")
    for col in target_cols:
        for lag in [1, 2, 3, 6, 12, 24, 48]:
            df[f'{col}_lag_{lag}'] = grp[col].shift(lag).astype('float32')
    print("OK")
    
    # ─── Diff features ───
    # Rozdiel oproti hodnote pred N krokmi - zachytáva trend (rastie/klesá hodnota?).
    # Pre 1h granularitu: 1 krok a 24 krokov (1 deň)
    print("  Diffs...", end=" ")
    for col in target_cols:
        for d in [1, 144]:
            df[f'{col}_diff_{d}'] = grp[col].diff(d).astype('float32')
    print("OK")
    
    # ─── Rolling mean ───
    # Kĺzavý priemer cez posledných N krokov - odhalí lokálnu úroveň/trend.
    # shift(1) najprv posunie - aby sme do priemeru nezahrnuli aktuálnu hodnotu (data leakage)
    print("  Rolling...", end=" ")
    for col in target_cols:
        shifted = grp[col].shift(1)
        for w in [6, 144]:
            df[f'{col}_rmean_{w}'] = shifted.rolling(w, min_periods=1).mean().astype('float32')
        del shifted
    print("OK")
    del grp; gc.collect()
    
    # Vyberieme len features (vylúčime metadata a targety)
    exclude = ['eic', 'segment_id', 't_utc', 'Elektromer', 'typ_siete', 'Chyba', 'valid_segment']
    feature_cols = sorted([c for c in df.columns if c not in exclude + target_cols])
    
    # Riadky, kde nejaký lag/rolling nemal dosť histórie, sú NaN - tie zahodíme
    df = df.dropna(subset=feature_cols).reset_index(drop=True)
    print(f"  Po dropna: {len(df):,}")
    
    # ─── Uloženie na disk ───
    # Metadata (eic, čas, Chyba) uložíme do parquet na neskoršie spárovanie predikcií
    meta = df[['eic', 'segment_id', 't_utc', 'Chyba']].copy()
    meta.to_parquet(f'{cache_dir}/{split_name}_meta.parquet', index=False)
    
    # X (features) uložíme ako numpy - rýchle načítanie cez mmap_mode neskôr
    X = df[feature_cols].values.astype('float32')
    np.save(f'{cache_dir}/{split_name}_X.npy', X)
    
    # Y (targety) - pre každú kombináciu (target × horizon) uložíme samostatný array.
    # shift(-h) posunie hodnotu o h krokov dopredu (to je to, čo predikujeme)
    grp2 = df.groupby(['eic', 'segment_id'])
    for target in target_cols:
        for h in horizons:
            y = grp2[target].shift(-h).loc[df.index].values.astype('float32')
            np.save(f'{cache_dir}/{split_name}_y_{target}_h{h}.npy', y)
    
    n0 = (meta['Chyba']==0).sum()
    n1 = (meta['Chyba']==1).sum()
    print(f"  X: {X.shape} ({X.nbytes/1024**3:.2f} GB)")
    print(f"  Chyba=0: {n0:,}, Chyba=1: {n1:,}")
    
    del df, meta, X, grp2; gc.collect()
    return feature_cols


In [ ]:
%%time
feature_cols = create_features_and_save(
    train_clean, 'train', TARGET_COLS, SELECTED_HORIZONS, CACHE_DIR)
del train_clean; gc.collect()
print(f"\nFeature stĺpce: {len(feature_cols)}")


In [ ]:
%%time
_ = create_features_and_save(
    val_clean, 'val', TARGET_COLS, SELECTED_HORIZONS, CACHE_DIR)
del val_clean; gc.collect()


In [ ]:
%%time
_ = create_features_and_save(
    test_clean, 'test', TARGET_COLS, SELECTED_HORIZONS, CACHE_DIR)
del test_clean; gc.collect()


In [ ]:
%%time
_ = create_features_and_save(
    anomaly_data, 'anomaly', TARGET_COLS, SELECTED_HORIZONS, CACHE_DIR)
del anomaly_data, error_data; gc.collect()


---
# ČASŤ 3: OPTUNA HYPERPARAMETER TUNING
---

In [ ]:
%%time
print("=" * 60)
print("OPTUNA HYPERPARAMETER TUNING")
print("=" * 60)

# mmap_mode='r' - numpy súbor sa namapuje do pamäte ale fyzicky sa nenačíta celý.
# Pri viacgigabajtových arrays toto výrazne šetrí RAM.
X_train_opt = np.load(f'{CACHE_DIR}/train_X.npy', mmap_mode='r')
X_val_opt   = np.load(f'{CACHE_DIR}/val_X.npy', mmap_mode='r')

y_train_opt = np.load(f'{CACHE_DIR}/train_y_{OPTUNA_TARGET}_h{OPTUNA_HORIZON}.npy')
y_val_opt   = np.load(f'{CACHE_DIR}/val_y_{OPTUNA_TARGET}_h{OPTUNA_HORIZON}.npy')

# Riadky bez targetu (NaN) zahodíme - to sú miesta blízko konca segmentu, kde shift(-h) zlyhá
train_m = ~np.isnan(y_train_opt)
val_m   = ~np.isnan(y_val_opt)

# np.array(...) urobí kópiu z mmap-ovaného array do RAM - LightGBM potrebuje skutočný array
X_tr = np.array(X_train_opt[train_m])
y_tr = y_train_opt[train_m]
X_vl = np.array(X_val_opt[val_m])
y_vl = y_val_opt[val_m]

print(f"Tuning na: {OPTUNA_TARGET} h{OPTUNA_HORIZON}")
print(f"Train: {X_tr.shape[0]:,}, Val (clean): {X_vl.shape[0]:,}")

def objective(trial):
    # Hlavné LightGBM hyperparametre - viď LightGBM docs pre detaily
    params = {
        **LGBM_BASE,
        'num_leaves':        trial.suggest_int('num_leaves', 15, 127),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    
    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    dval   = lgb.Dataset(X_vl, label=y_vl, reference=dtrain, free_raw_data=False)
    
    # Early stopping zastaví trénovanie, ak sa val MAE 50 kôl nezlepší
    model = lgb.train(
        params, dtrain,
        num_boost_round=NUM_BOOST_ROUNDS,
        valid_sets=[dval],
        valid_names=['val'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )
    
    pred = model.predict(X_vl)
    return mean_absolute_error(y_vl, pred)

study = optuna.create_study(direction='minimize', study_name='lgbm_tuning')
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

print(f"\nNajlepšia MAE: {study.best_value:.6f}")
print(f"Najlepšie parametre:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# Spojíme base parametre s najlepšími z Optuny - tieto použijeme pre všetkých 6 cieľov
LGBM_PARAMS = {**LGBM_BASE, **study.best_params}
print(f"\nLGBM_PARAMS nastavené z Optuna.")

del X_train_opt, X_val_opt, y_train_opt, y_val_opt, X_tr, y_tr, X_vl, y_vl
gc.collect()


---
# ČASŤ 4: TRÉNOVANIE A FORECASTING METRIKY
---

In [ ]:
%%time
print("=" * 60)
print("TRÉNOVANIE LightGBM (forecasting na čistých dátach)")
print("=" * 60)

# Načítanie features cez mmap_mode (šetrí RAM)
X_train = np.load(f'{CACHE_DIR}/train_X.npy', mmap_mode='r')
X_val   = np.load(f'{CACHE_DIR}/val_X.npy', mmap_mode='r')
X_test  = np.load(f'{CACHE_DIR}/test_X.npy', mmap_mode='r')
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

meta_val  = pd.read_parquet(f'{CACHE_DIR}/val_meta.parquet')
meta_test = pd.read_parquet(f'{CACHE_DIR}/test_meta.parquet')

models = {}
forecast_results = []

# Pre každú kombináciu (target × horizon) trénujeme samostatný model.
# Pri 6 targetoch a 4 horizontoch = 24 modelov. Všetky majú rovnaké hyperparametre z Optuny.
for target in TARGET_COLS:
    print(f"\nTarget: {target}")
    for horizon in SELECTED_HORIZONS:
        y_train = np.load(f'{CACHE_DIR}/train_y_{target}_h{horizon}.npy')
        y_val   = np.load(f'{CACHE_DIR}/val_y_{target}_h{horizon}.npy')
        y_test  = np.load(f'{CACHE_DIR}/test_y_{target}_h{horizon}.npy')

        # Maska na riadky kde target nie je NaN
        train_mask = ~np.isnan(y_train)
        val_mask   = ~np.isnan(y_val)
        test_mask  = ~np.isnan(y_test)

        dtrain = lgb.Dataset(np.array(X_train[train_mask]),
                             label=y_train[train_mask], free_raw_data=True)
        dval = lgb.Dataset(np.array(X_val[val_mask]),
                           label=y_val[val_mask], reference=dtrain, free_raw_data=True)

        model = lgb.train(
            LGBM_PARAMS, dtrain,
            num_boost_round=NUM_BOOST_ROUNDS,
            valid_sets=[dtrain, dval],
            valid_names=['train', 'val'],
            callbacks=[
                lgb.early_stopping(stopping_rounds=EARLY_STOPPING, verbose=False),
                lgb.log_evaluation(period=0)
            ]
        )
        models[(target, horizon)] = model

        val_pred  = model.predict(np.array(X_val[val_mask]))
        test_pred = model.predict(np.array(X_test[test_mask]))

        print(f"  h{horizon:2d}: val_MAE={mean_absolute_error(y_val[val_mask], val_pred):.4f}, "
              f"test_MAE={mean_absolute_error(y_test[test_mask], test_pred):.4f}, trees={model.best_iteration}")

        # Spárujeme predikcie s metadátami (eic, čas, Chyba) cez masku
        for split_name, meta_df, y_true, y_pred, mask in [
            ('val', meta_val, y_val, val_pred, val_mask),
            ('test', meta_test, y_test, test_pred, test_mask)
        ]:
            batch = pd.DataFrame({
                'eic': meta_df['eic'].values[mask],
                'segment_id': meta_df['segment_id'].values[mask],
                't_utc': meta_df['t_utc'].values[mask],
                'Chyba': meta_df['Chyba'].values[mask],
                'target': target,
                'hour_ahead': horizon,
                'prediction': y_pred.astype('float32'),
                'actual': y_true[mask].astype('float32'),
                'split': split_name
            })
            forecast_results.append(batch)
        del dtrain, dval, y_train, y_val, y_test; gc.collect()

forecast_df = pd.concat(forecast_results, ignore_index=True)
forecast_df['residual'] = forecast_df['actual'] - forecast_df['prediction']
forecast_df['abs_error'] = np.abs(forecast_df['residual'])
del forecast_results; gc.collect()

print(f"\nModelov: {len(models)}, Predikcií (čisté): {len(forecast_df):,}")


In [ ]:
# FORECASTING METRIKY (len Chyba=0 dáta)
for split in ['val', 'test']:
    split_label = 'VALIDATION (Chyba=0)' if split == 'val' else 'TEST (Chyba=0)'
    sd = forecast_df[forecast_df['split'] == split]
    
    print(f"\n{'='*60}")
    print(f"  {split_label}: {len(sd):,} predikcií")
    print(f"{'='*60}")
    
    # Napätie
    print(f"\n  NAPÄTIE (u1, u2, u3):")
    rows_v = []
    for t in VOLTAGE_COLS:
        d = sd[sd['target'] == t].dropna(subset=['actual', 'prediction'])
        rows_v.append({
            'target': t,
            'MAE': mean_absolute_error(d['actual'], d['prediction']),
            'RMSE': np.sqrt(mean_squared_error(d['actual'], d['prediction'])),
            'R2': r2_score(d['actual'], d['prediction'])
        })
    mv = pd.DataFrame(rows_v)
    print(mv.to_string(index=False))
    print(f"    → Priemer napätie: MAE={mv['MAE'].mean():.4f}, RMSE={mv['RMSE'].mean():.4f}, R²={mv['R2'].mean():.4f}")
    
    # Prúd
    print(f"\n  PRÚDY (i1, i2, i3):")
    rows_c = []
    for t in CURRENT_COLS:
        d = sd[sd['target'] == t].dropna(subset=['actual', 'prediction'])
        rows_c.append({
            'target': t,
            'MAE': mean_absolute_error(d['actual'], d['prediction']),
            'RMSE': np.sqrt(mean_squared_error(d['actual'], d['prediction'])),
            'R2': r2_score(d['actual'], d['prediction'])
        })
    mc = pd.DataFrame(rows_c)
    print(mc.to_string(index=False))
    print(f"    → Priemer prúdy:   MAE={mc['MAE'].mean():.4f}, RMSE={mc['RMSE'].mean():.4f}, R²={mc['R2'].mean():.4f}")


In [ ]:
# Ako klesá presnosť so vzdialenosťou predikcie - kratšie horizonty by mali byť presnejšie
print("METRIKY PODĽA HORIZONTU (test, Chyba=0):")
sd = forecast_df[forecast_df['split'] == 'test']

print("\n  Napätie:")
for h in SELECTED_HORIZONS:
    d = sd[(sd['hour_ahead']==h) & (sd['target'].isin(VOLTAGE_COLS))]
    if len(d) == 0: continue
    print(f"    h{h:2d}: MAE={mean_absolute_error(d['actual'], d['prediction']):.4f}, "
          f"RMSE={np.sqrt(mean_squared_error(d['actual'], d['prediction'])):.4f}, "
          f"R²={r2_score(d['actual'], d['prediction']):.4f}")

print("\n  Prúdy:")
for h in SELECTED_HORIZONS:
    d = sd[(sd['hour_ahead']==h) & (sd['target'].isin(CURRENT_COLS))]
    if len(d) == 0: continue
    print(f"    h{h:2d}: MAE={mean_absolute_error(d['actual'], d['prediction']):.4f}, "
          f"RMSE={np.sqrt(mean_squared_error(d['actual'], d['prediction'])):.4f}, "
          f"R²={r2_score(d['actual'], d['prediction']):.4f}")


In [ ]:
# Feature importance - aké features model najviac používa pri rozhodovaní.
# Ukážka pre jeden model (u1_norm, najkratší horizont) - typicky bývajú najdôležitejšie
# nedávne lagy a kĺzavé priemery.
first_h = SELECTED_HORIZONS[0]
sample_model = models[('u1_norm', first_h)]
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': sample_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top = importance.head(20)
ax.barh(range(len(top)), top['importance'].values, color='#5B4F9E', alpha=0.85)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top['feature'].values)
ax.invert_yaxis()  # najdôležitejší feature hore
ax.set_title(f'Top 20 Features (u1_norm, h{first_h})', fontsize=14)
ax.set_xlabel('Gain')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


---
# ČASŤ 5: ANOMALY DETECTION
---

In [ ]:
%%time
print("=" * 60)
print("ANOMALY DETECTION (model trénovaný na čistých, testovaný na všetkých)")
print("="*60)

# ─────────────────────────────────────────────────
# KROK 1: Predikcie na VŠETKÝCH dátach (Chyba=0 + Chyba=1)
# ─────────────────────────────────────────────────
X_anomaly = np.load(f'{CACHE_DIR}/anomaly_X.npy', mmap_mode='r')
meta_anomaly = pd.read_parquet(f'{CACHE_DIR}/anomaly_meta.parquet')
print(f"Anomaly dáta: {X_anomaly.shape[0]:,} riadkov")
print(f"  Chyba=0: {(meta_anomaly['Chyba']==0).sum():,}, Chyba=1: {(meta_anomaly['Chyba']==1).sum():,}")

# Pre každý uložený model spustíme predikciu na anomaly dátach
anomaly_results = []
for target in TARGET_COLS:
    for horizon in SELECTED_HORIZONS:
        y_anom = np.load(f'{CACHE_DIR}/anomaly_y_{target}_h{horizon}.npy')
        mask = ~np.isnan(y_anom)
        
        model = models[(target, horizon)]
        pred = model.predict(np.array(X_anomaly[mask]))
        
        batch = pd.DataFrame({
            'eic': meta_anomaly['eic'].values[mask],
            'segment_id': meta_anomaly['segment_id'].values[mask],
            't_utc': meta_anomaly['t_utc'].values[mask],
            'Chyba': meta_anomaly['Chyba'].values[mask],
            'target': target,
            'hour_ahead': horizon,
            'prediction': pred.astype('float32'),
            'actual': y_anom[mask].astype('float32'),
        })
        anomaly_results.append(batch)
        del y_anom; gc.collect()

anomaly_df = pd.concat(anomaly_results, ignore_index=True)
anomaly_df['abs_error'] = np.abs(anomaly_df['actual'] - anomaly_df['prediction'])
del anomaly_results, X_anomaly; gc.collect()

print(f"Anomaly predikcií: {len(anomaly_df):,}")

# ─────────────────────────────────────────────────
# KROK 2: Anomaly score = priemerná |chyba| cez 6 targetov
# ─────────────────────────────────────────────────
# Vyšší score = predikcia sa viac líši od skutočnosti = pravdepodobnejšia anomália
anom_agg = anomaly_df.groupby(['eic', 'segment_id', 't_utc', 'hour_ahead'], observed=True).agg(
    Chyba=('Chyba', 'first'),
    anomaly_score=('abs_error', 'mean')
).reset_index()

print(f"\nanom_agg: {len(anom_agg):,} (Chyba=0: {(anom_agg['Chyba']==0).sum():,}, Chyba=1: {(anom_agg['Chyba']==1).sum():,})")

# ─────────────────────────────────────────────────
# KROK 3: Prah z ČISTÝCH dát (Chyba=0)
#   Model bol trénovaný na normálnych → residuály na normálnych = baseline
# ─────────────────────────────────────────────────
clean_scores = anom_agg[anom_agg['Chyba'] == 0]['anomaly_score'].values
threshold = clean_scores.mean() + ANOMALY_SIGMA * clean_scores.std()
print(f"\nPrah (mean + {ANOMALY_SIGMA}σ z čistých dát): {threshold:.4f}")
print(f"  mean = {clean_scores.mean():.4f}, std = {clean_scores.std():.4f}")

# ─────────────────────────────────────────────────
# KROK 4: Detekcia na CELOM datasete
# ─────────────────────────────────────────────────
anom_agg['predicted_anomaly'] = (anom_agg['anomaly_score'] > threshold).astype(int)

y_true = anom_agg['Chyba'].values
y_pred = anom_agg['predicted_anomaly'].values
scores = anom_agg['anomaly_score'].values

# Confusion matrix - .ravel() rozbalí maticu 2x2 na 4 čísla [tn, fp, fn, tp]
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)

print(f"\nCONFUSION MATRIX:")
print(f"                  Predikované")
print(f"                Normal  Anomália")
print(f"  Skut. Normal  {tn:>7,}  {fp:>7,}")
print(f"  Skut. Chyba   {fn:>7,}  {tp:>7,}")
print(f"\n  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  F1-score:  {f1:.4f}")

# ROC-AUC a PR-AUC nepotrebujú prah - pracujú priamo s anomaly_score
try:
    auc_roc = roc_auc_score(y_true, scores)
    pr_auc  = average_precision_score(y_true, scores)
    print(f"  ROC-AUC:   {auc_roc:.4f}")
    print(f"  PR-AUC:    {pr_auc:.4f}")
except:
    auc_roc, pr_auc = None, None

# Metriky podľa horizontu
print(f"\nPODĽA HORIZONTU:")
for h in SELECTED_HORIZONS:
    hd = anom_agg[anom_agg['hour_ahead']==h]
    hp = precision_score(hd['Chyba'], hd['predicted_anomaly'], zero_division=0)
    hr = recall_score(hd['Chyba'], hd['predicted_anomaly'], zero_division=0)
    hf = f1_score(hd['Chyba'], hd['predicted_anomaly'], zero_division=0)
    print(f"  h{h:2d}: Precision={hp:.4f}, Recall={hr:.4f}, F1={hf:.4f}")


In [ ]:
# Citlivostná analýza prahu - skúsime rôzne σ a pozrieme, ako sa menia precision/recall.
# Pomáha vidieť trade-off: nižšie σ = viac true positives ale aj viac false positives.
print("THRESHOLD ANALÝZA:")
print(f"{'σ':>5s} {'Prec':>8s} {'Recall':>8s} {'F1':>8s} {'TP':>7s} {'FP':>7s} {'FN':>7s}")

sigma_results = []
for sigma in [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    th = clean_scores.mean() + sigma * clean_scores.std()
    yp = (scores > th).astype(int)
    sp = precision_score(y_true, yp, zero_division=0)
    sr = recall_score(y_true, yp, zero_division=0)
    sf = f1_score(y_true, yp, zero_division=0)
    s_cm = confusion_matrix(y_true, yp, labels=[0,1])
    _, sfp, sfn, stp = s_cm.ravel()
    sigma_results.append({'σ': sigma, 'P': sp, 'R': sr, 'F1': sf})
    print(f"{sigma:5.1f} {sp:8.4f} {sr:8.4f} {sf:8.4f} {stp:7d} {sfp:7d} {sfn:7d}")

# Vykreslenie ako funkcie σ - vidíme, kde sa P, R, F1 "prelomia"
sr_df = pd.DataFrame(sigma_results)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sr_df['σ'], sr_df['P'], 'b-o', label='Precision', linewidth=2)
ax.plot(sr_df['σ'], sr_df['R'], 'r-s', label='Recall', linewidth=2)
ax.plot(sr_df['σ'], sr_df['F1'], 'g-^', label='F1', linewidth=2)
ax.set_xlabel('σ multiplikátor', fontsize=12)
ax.set_ylabel('Skóre', fontsize=12)
ax.set_title('Threshold analýza – Precision / Recall / F1', fontsize=14)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


---
# ČASŤ 6: VIZUALIZÁCIE
---

In [ ]:
# Reálne vs Predikované – NAPÄTIE (test, Chyba=0)
# Diagonála = perfektná predikcia, body čím bližšie k nej tým lepšie
td = forecast_df[forecast_df['split'] == 'test']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for i, t in enumerate(VOLTAGE_COLS):
    ax = axes[i]
    d = td[td['target'] == t].dropna(subset=['actual', 'prediction'])
    r2 = r2_score(d['actual'], d['prediction'])
    
    # Pri väčšom počte bodov sample-neme - 5000 stačí na dobrý vizuál a kreslí sa rýchlejšie
    if len(d) > 5000: d = d.sample(5000, random_state=42)
    
    ax.scatter(d['actual'], d['prediction'], alpha=0.12, s=3, c='#5B4F9E')
    # Diagonálna čiara od min do max - ideálna predikcia
    lims = [d[['actual','prediction']].min().min(), d[['actual','prediction']].max().max()]
    ax.plot(lims, lims, 'k--', alpha=0.4, linewidth=1)
    ax.set_xlabel('Skutočné [V]')
    ax.set_ylabel('Predikované [V]')
    ax.set_title(f'{t}  (R² = {r2:.4f})')

plt.suptitle('Reálne vs Predikované – Napätie (test, Chyba=0)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/scatter_voltage.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Reálne vs Predikované – PRÚDY (test, Chyba=0)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for i, t in enumerate(CURRENT_COLS):
    ax = axes[i]
    d = td[td['target'] == t].dropna(subset=['actual', 'prediction'])
    r2 = r2_score(d['actual'], d['prediction'])
    if len(d) > 5000: d = d.sample(5000, random_state=42)
    
    ax.scatter(d['actual'], d['prediction'], alpha=0.12, s=3, c='#D4582A')
    lims = [d[['actual','prediction']].min().min(), d[['actual','prediction']].max().max()]
    ax.plot(lims, lims, 'k--', alpha=0.4, linewidth=1)
    ax.set_xlabel('Skutočné [A]')
    ax.set_ylabel('Predikované [A]')
    ax.set_title(f'{t}  (R² = {r2:.4f})')

plt.suptitle('Reálne vs Predikované – Prúdy (test, Chyba=0)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/scatter_current.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Anomaly score v čase s prahom - vyberieme EIC s najviac poruchovými záznamami,
# aby sme videli, ako reziduály "vystrelia" počas reálnej poruchy
mid_h = SELECTED_HORIZONS[len(SELECTED_HORIZONS)//2]
fault_eics = anom_agg[anom_agg['Chyba']==1]['eic'].unique()

if len(fault_eics) > 0:
    # EIC s najviac poruchovými záznamami - tam bude najjasnejší "príbeh"
    best_eic = anom_agg[anom_agg['Chyba']==1].groupby('eic').size().idxmax()
    eic_data = anom_agg[(anom_agg['eic']==best_eic) & (anom_agg['hour_ahead']==mid_h)].sort_values('t_utc')
    
    fig, ax = plt.subplots(figsize=(14, 4))
    times = pd.to_datetime(eic_data['t_utc'])
    
    ax.plot(times, eic_data['anomaly_score'], color='#333', linewidth=0.6, alpha=0.8)
    # Horizontálna čiara = prah anomálie
    ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1.2, label=f'Prah ({ANOMALY_SIGMA}σ)')
    
    # Vyfarbené pásmo = obdobia, kedy bola skutočná porucha
    fault_mask = eic_data['Chyba'].values == 1
    if fault_mask.any():
        ymax = max(eic_data['anomaly_score'].max() * 1.2, threshold * 1.3)
        ax.fill_between(times, 0, ymax, where=fault_mask, alpha=0.10, color='red', label='Skutočná porucha')
    
    ax.set_xlabel('Čas')
    ax.set_ylabel('Anomaly score')
    ax.set_title(f'Anomaly score – EIC: {best_eic} (h{mid_h})')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/anomaly_score_time.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Žiadne poruchové dáta")


In [ ]:
# ČASOVÝ PRIEBEH – NAPÄTIE (u1_norm) – náhodný EIC, ukážka okna predikcií
mid_h = SELECTED_HORIZONS[0]
td = forecast_df[(forecast_df['split'] == 'test') & (forecast_df['target'] == 'u1_norm') & 
                  (forecast_df['hour_ahead'] == mid_h)].dropna(subset=['actual', 'prediction'])

WINDOW = 72  # počet bodov v zobrazenom okne

# Vyberieme len EIC, kde R² je dosť vysoké - ukážka má byť reprezentatívna
eic_r2 = {}
for eic, grp in td.groupby('eic'):
    if len(grp) >= WINDOW:
        r2 = r2_score(grp['actual'], grp['prediction'])
        if r2 > 0.5:
            eic_r2[eic] = r2

if len(eic_r2) > 0:
    # Náhodný výber jedného z dobrých EIC
    np.random.seed(1)
    selected_eic = np.random.choice(list(eic_r2.keys()))
    eic_data = td[td['eic'] == selected_eic].sort_values('t_utc').reset_index(drop=True)
    
    # Náhodné okno z časového radu - vyberieme náhodný úsek dĺžky WINDOW
    start_idx = np.random.randint(0, max(1, len(eic_data) - WINDOW))
    window = eic_data.iloc[start_idx:start_idx + WINDOW]
    
    t = pd.to_datetime(window['t_utc'])
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, window['actual'], color='#1f77b4', linewidth=1.5, marker='o', markersize=3, label='Skutočné')
    ax.plot(t, window['prediction'], color='#ff7f0e', linewidth=1.5, marker='s', markersize=3, label='Predikované')
    ax.set_ylabel('U1 [V]')
    ax.set_xlabel('Čas')
    ax.set_title(f'Predikcia napätia U1 – EIC: {selected_eic} (h{mid_h}, R²={eic_r2[selected_eic]:.4f})')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/timeseries_voltage_forecast.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ČASOVÝ PRIEBEH – PRÚD (i1_norm) – náhodný EIC, ukážka okna predikcií
td_i = forecast_df[(forecast_df['split'] == 'test') & (forecast_df['target'] == 'i1_norm') & 
                    (forecast_df['hour_ahead'] == mid_h)].dropna(subset=['actual', 'prediction'])

WINDOW = 72

eic_r2_i = {}
for eic, grp in td_i.groupby('eic'):
    if len(grp) >= WINDOW:
        r2 = r2_score(grp['actual'], grp['prediction'])
        if r2 > 0.5:
            eic_r2_i[eic] = r2

if len(eic_r2_i) > 0:
    # Iný seed (8) - aby sa nevybral ten istý EIC ako pri napätí
    np.random.seed(8)
    selected_eic = np.random.choice(list(eic_r2_i.keys()))
    eic_data = td_i[td_i['eic'] == selected_eic].sort_values('t_utc').reset_index(drop=True)
    
    start_idx = np.random.randint(0, max(1, len(eic_data) - WINDOW))
    window = eic_data.iloc[start_idx:start_idx + WINDOW]
    
    t = pd.to_datetime(window['t_utc'])
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, window['actual'], color='#1f77b4', linewidth=1.5, marker='o', markersize=3, label='Skutočné')
    ax.plot(t, window['prediction'], color='#ff7f0e', linewidth=1.5, marker='s', markersize=3, label='Predikované')
    ax.set_ylabel('I1 [A]')
    ax.set_xlabel('Čas')
    ax.set_title(f'Predikcia prúdu I1 – EIC: {selected_eic} (h{mid_h}, R²={eic_r2_i[selected_eic]:.4f})')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/timeseries_current_forecast.png', dpi=150, bbox_inches='tight')
    plt.show()


---
# ČASŤ 7: EXPORT
---

In [ ]:
# Uloženie predikcií, feature importance a všetkých 24 modelov
forecast_df.to_csv(f'{OUTPUT_DIR}/lightgbm_forecast_results.csv', sep=';', index=False)
importance.to_csv(f'{OUTPUT_DIR}/lightgbm_feature_importance.csv', sep=';', index=False)
anom_agg.to_csv(f'{OUTPUT_DIR}/lightgbm_anomaly_results.csv', sep=';', index=False)

# Modely uložíme do textových súborov - LightGBM má vlastný formát ktorý sa dá načítať späť
for (target, horizon), model in models.items():
    model.save_model(f'{OUTPUT_DIR}/lgbm_{target}_h{horizon}.txt')

print(f"Uložené do: {OUTPUT_DIR}/")
print(f"  - lightgbm_forecast_results.csv (forecasting na čistých)")
print(f"  - lightgbm_anomaly_results.csv (detekcia na zmiešaných)")
print(f"  - lightgbm_feature_importance.csv")
print(f"  - {len(models)} model súborov")
